In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from src.data.loader import load_sales_data
from src.preprocessing.sales import prepare_sales_data

SALES_PATH = PROJECT_ROOT / "data" / "raw" / "sales.csv"

df = load_sales_data(SALES_PATH)
df = prepare_sales_data(df)

df.head()

,date,product_id,product_name,category,price,discount,promotion,day_of_week,month,is_weekend,units_sold
0,2024-01-01,P001,Wireless Headphones,Electronics,1999.0,0,0,0,1,False,33
1,2024-01-02,P001,Wireless Headphones,Electronics,1999.0,0,0,1,1,False,39
2,2024-01-03,P001,Wireless Headphones,Electronics,1999.0,0,0,2,1,False,39
3,2024-01-04,P001,Wireless Headphones,Electronics,1799.1,10,1,3,1,False,59
4,2024-01-05,P001,Wireless Headphones,Electronics,1999.0,0,0,4,1,False,35


In [3]:
from src.features.time_features import create_time_features
from src.features.lag_features import create_lag_features
from src.features.rolling_features import create_rolling_features

In [4]:
df = create_time_features(df)
df = create_lag_features(df)
df = create_rolling_features(df)

df.tail()

,date,product_id,product_name,category,price,discount,promotion,day_of_week,month,is_weekend,...,day,week_of_year,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_14,rolling_mean_28,rolling_std_7
3650,2025-12-27,P005,Yoga Mat,Fitness,999.0,0,0,5,12,True,...,27,52,35.0,64.0,49.0,54.0,54.428571,51.500000,52.285714,11.133390
3651,2025-12-28,P005,Yoga Mat,Fitness,999.0,0,0,6,12,True,...,28,52,43.0,49.0,84.0,54.0,51.428571,51.071429,51.892857,10.952277
3652,2025-12-29,P005,Yoga Mat,Fitness,999.0,0,0,0,12,False,...,29,1,43.0,59.0,37.0,32.0,50.571429,48.142857,51.500000,11.399666
3653,2025-12-30,P005,Yoga Mat,Fitness,999.0,0,0,1,12,False,...,30,1,24.0,47.0,33.0,50.0,45.571429,47.214286,51.214286,14.374249
3654,2025-12-31,P005,Yoga Mat,Fitness,999.0,0,0,2,12,False,...,31,1,46.0,65.0,44.0,44.0,45.428571,48.142857,51.071429,14.362650


In [5]:
import joblib

MODEL_PATH = PROJECT_ROOT / "models" / "xgboost_forecaster.joblib"

model = joblib.load(MODEL_PATH)

print("XGBoost model loaded successfully.")

XGBoost model loaded successfully.


In [6]:
INVENTORY_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "inventory_snapshot.csv"
)

inventory_df = pd.read_csv(
    INVENTORY_PATH,
    parse_dates=[
        "snapshot_date",
        "expected_arrival_date"
    ]
)

inventory_df

,snapshot_date,product_id,current_stock,open_order_qty,expected_arrival_date,lead_time_days,unit_cost
0,2025-12-31,P001,225,0,NaT,4,1000
1,2025-12-31,P002,190,0,NaT,7,1800
2,2025-12-31,P003,191,0,NaT,5,2500
3,2025-12-31,P004,202,0,NaT,3,1200
4,2025-12-31,P005,251,0,NaT,6,700


In [7]:
print(df.tail())
print(inventory_df)

           date product_id product_name category  price  discount  promotion  \
3650 2025-12-27       P005     Yoga Mat  Fitness  999.0         0          0   
3651 2025-12-28       P005     Yoga Mat  Fitness  999.0         0          0   
3652 2025-12-29       P005     Yoga Mat  Fitness  999.0         0          0   
3653 2025-12-30       P005     Yoga Mat  Fitness  999.0         0          0   
3654 2025-12-31       P005     Yoga Mat  Fitness  999.0         0          0   

      day_of_week  month  is_weekend  ...  day  week_of_year  lag_1  lag_7  \
3650            5     12        True  ...   27            52   35.0   64.0   
3651            6     12        True  ...   28            52   43.0   49.0   
3652            0     12       False  ...   29             1   43.0   59.0   
3653            1     12       False  ...   30             1   24.0   47.0   
3654            2     12       False  ...   31             1   46.0   65.0   

      lag_14  lag_28  rolling_mean_7  rolling_mean

In [8]:
MODEL_FEATURES = [
    "price",
    "discount",
    "promotion",
    "day_of_week",
    "month",
    "is_weekend",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7",
]

In [9]:
def forecast_product_demand(
    model,
    product_history,
    horizon=30
):
    """
    Generate recursive future demand forecasts
    for one product.
    """

    history = product_history.copy()

    history = history.sort_values("date").reset_index(drop=True)

    forecasts = []

    for _ in range(horizon):

        next_date = history["date"].max() + pd.Timedelta(days=1)

        # --------------------------------------
        # Calendar features
        # --------------------------------------

        day_of_week = next_date.dayofweek
        month = next_date.month
        is_weekend = day_of_week >= 5

        # --------------------------------------
        # Future business assumptions
        # --------------------------------------

        last_price = history["price"].iloc[-1]

        price = last_price
        discount = 0
        promotion = 0

        # --------------------------------------
        # Historical demand series
        # --------------------------------------

        demand_history = history["units_sold"]

        # --------------------------------------
        # Lag features
        # --------------------------------------

        lag_1 = demand_history.iloc[-1]
        lag_7 = demand_history.iloc[-7]
        lag_14 = demand_history.iloc[-14]
        lag_28 = demand_history.iloc[-28]

        # --------------------------------------
        # Rolling features
        # --------------------------------------

        rolling_mean_7 = demand_history.iloc[-7:].mean()
        rolling_mean_14 = demand_history.iloc[-14:].mean()
        rolling_mean_28 = demand_history.iloc[-28:].mean()

        rolling_std_7 = demand_history.iloc[-7:].std()

        # --------------------------------------
        # Create model input
        # --------------------------------------

        X_future = pd.DataFrame([{
            "price": price,
            "discount": discount,
            "promotion": promotion,
            "day_of_week": day_of_week,
            "month": month,
            "is_weekend": is_weekend,
            "lag_1": lag_1,
            "lag_7": lag_7,
            "lag_14": lag_14,
            "lag_28": lag_28,
            "rolling_mean_7": rolling_mean_7,
            "rolling_mean_14": rolling_mean_14,
            "rolling_mean_28": rolling_mean_28,
            "rolling_std_7": rolling_std_7,
        }])

        # --------------------------------------
        # Predict demand
        # --------------------------------------

        prediction = model.predict(
            X_future[MODEL_FEATURES]
        )[0]

        prediction = max(0, prediction)

        # --------------------------------------
        # Add prediction to history
        # --------------------------------------

        future_row = {
            "date": next_date,
            "product_id": history["product_id"].iloc[0],
            "product_name": history["product_name"].iloc[0],
            "category": history["category"].iloc[0],
            "price": price,
            "discount": discount,
            "promotion": promotion,
            "units_sold": prediction,
        }

        history = pd.concat(
            [
                history,
                pd.DataFrame([future_row])
            ],
            ignore_index=True
        )

        forecasts.append({
            "date": next_date,
            "product_id": history["product_id"].iloc[0],
            "forecast_units": prediction
        })

    return pd.DataFrame(forecasts)

In [10]:
p001_history = df[
    df["product_id"] == "P001"
].copy()

p001_forecast = forecast_product_demand(
    model=model,
    product_history=p001_history,
    horizon=30
)

p001_forecast

,date,product_id,forecast_units
0,2026-01-01,P001,36.979706
1,2026-01-02,P001,37.337029
2,2026-01-03,P001,42.004692
3,2026-01-04,P001,41.931583
4,2026-01-05,P001,35.965870
5,2026-01-06,P001,34.924202
6,2026-01-07,P001,35.043671
7,2026-01-08,P001,34.926353
8,2026-01-09,P001,35.163754
9,2026-01-10,P001,39.377102


In [11]:
print(
    f"P001 30-day forecast: "
    f"{p001_forecast['forecast_units'].sum():.0f} units"
)

P001 30-day forecast: 1045 units


In [12]:
p001_forecast.describe()

,date,forecast_units
count,30,30.000000
mean,2026-01-15 12:00:00,34.829964
min,2026-01-01 00:00:00,29.120676
25%,2026-01-08 06:00:00,31.988782
50%,2026-01-15 12:00:00,34.902672
75%,2026-01-22 18:00:00,36.726247
max,2026-01-30 00:00:00,42.004692
std,NaN,3.635933


In [13]:
all_forecasts = []

for product_id in df["product_id"].unique():

    product_history = df[
        df["product_id"] == product_id
    ].copy()

    product_forecast = forecast_product_demand(
        model=model,
        product_history=product_history,
        horizon=30
    )

    all_forecasts.append(product_forecast)


forecast_df = pd.concat(
    all_forecasts,
    ignore_index=True
)

forecast_df.head()


,date,product_id,forecast_units
0,2026-01-01,P001,36.979706
1,2026-01-02,P001,37.337029
2,2026-01-03,P001,42.004692
3,2026-01-04,P001,41.931583
4,2026-01-05,P001,35.965870


In [14]:
forecast_df.tail()

,date,product_id,forecast_units
145,2026-01-26,P005,34.144382
146,2026-01-27,P005,34.144382
147,2026-01-28,P005,34.144382
148,2026-01-29,P005,33.872223
149,2026-01-30,P005,33.752140


In [15]:
forecast_summary = (
    forecast_df
    .groupby("product_id")["forecast_units"]
    .agg(
        total_forecast="sum",
        avg_daily_forecast="mean",
        min_daily_forecast="min",
        max_daily_forecast="max"
    )
    .reset_index()
)

forecast_summary

,product_id,total_forecast,avg_daily_forecast,min_daily_forecast,max_daily_forecast
0,P001,1044.898804,34.829960,29.120676,42.004692
1,P002,781.246094,26.041536,23.081276,30.565094
2,P003,877.865967,29.262199,23.493504,36.795311
3,P004,620.698303,20.689943,19.717688,23.334322
4,P005,1156.417969,38.547264,33.752140,48.620548


In [16]:
# Create the latest historical state for each product

latest_state = (
    df
    .sort_values(["product_id", "date"])
    .groupby("product_id")
    .tail(1)
    [
        [
            "product_id",
            "date",
            "product_name",
            "category",
            "units_sold"
        ]
    ]
    .rename(
        columns={
            "date": "last_actual_date",
            "units_sold": "last_actual_demand"
        }
    )
    .reset_index(drop=True)
)

latest_state

,product_id,last_actual_date,product_name,category,last_actual_demand
0,P001,2025-12-31,Wireless Headphones,Electronics,24
1,P002,2025-12-31,Running Shoes,Footwear,22
2,P003,2025-12-31,Smart Watch,Electronics,33
3,P004,2025-12-31,Travel Backpack,Accessories,23
4,P005,2025-12-31,Yoga Mat,Fitness,33


In [17]:
# Combine latest historical state with current inventory

inventory_state = latest_state.merge(
    inventory_df,
    on="product_id",
    how="left"
)

inventory_state

,product_id,last_actual_date,product_name,category,last_actual_demand,snapshot_date,current_stock,open_order_qty,expected_arrival_date,lead_time_days,unit_cost
0,P001,2025-12-31,Wireless Headphones,Electronics,24,2025-12-31,225,0,NaT,4,1000
1,P002,2025-12-31,Running Shoes,Footwear,22,2025-12-31,190,0,NaT,7,1800
2,P003,2025-12-31,Smart Watch,Electronics,33,2025-12-31,191,0,NaT,5,2500
3,P004,2025-12-31,Travel Backpack,Accessories,23,2025-12-31,202,0,NaT,3,1200
4,P005,2025-12-31,Yoga Mat,Fitness,33,2025-12-31,251,0,NaT,6,700


In [18]:
# Attach lead time to each forecasted day

forecast_with_inventory = forecast_df.merge(
    inventory_state[
        [
            "product_id",
            "lead_time_days"
        ]
    ],
    on="product_id",
    how="left"
)

forecast_with_inventory.head(10)

,date,product_id,forecast_units,lead_time_days
0,2026-01-01,P001,36.979706,4
1,2026-01-02,P001,37.337029,4
2,2026-01-03,P001,42.004692,4
3,2026-01-04,P001,41.931583,4
4,2026-01-05,P001,35.965870,4
5,2026-01-06,P001,34.924202,4
6,2026-01-07,P001,35.043671,4
7,2026-01-08,P001,34.926353,4
8,2026-01-09,P001,35.163754,4
9,2026-01-10,P001,39.377102,4


In [19]:
# Calculate expected demand during supplier lead time

lead_time_demand = (
    forecast_with_inventory
    .sort_values(["product_id", "date"])
    .groupby("product_id")
    .apply(
        lambda group: group[
            group["date"]
            <= group["date"].min()
            + pd.Timedelta(
                days=group["lead_time_days"].iloc[0] - 1
            )
        ]["forecast_units"].sum()
    )
    .reset_index(name="lead_time_demand")
)

lead_time_demand

,product_id,lead_time_demand
0,P001,158.253006
1,P002,199.272079
2,P003,168.577713
3,P004,64.259758
4,P005,249.945099


In [20]:
ERROR_STD_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "forecast_error_std.csv"
)

error_std_df = pd.read_csv(ERROR_STD_PATH)

error_std_df

,product_id,error_std
0,P001,5.594148
1,P002,3.845135
2,P003,4.801437
3,P004,2.772442
4,P005,6.277869


In [21]:
inventory_state = inventory_state.merge(
    error_std_df,
    on="product_id",
    how="left"
)

inventory_state

,product_id,last_actual_date,product_name,category,last_actual_demand,snapshot_date,current_stock,open_order_qty,expected_arrival_date,lead_time_days,unit_cost,error_std
0,P001,2025-12-31,Wireless Headphones,Electronics,24,2025-12-31,225,0,NaT,4,1000,5.594148
1,P002,2025-12-31,Running Shoes,Footwear,22,2025-12-31,190,0,NaT,7,1800,3.845135
2,P003,2025-12-31,Smart Watch,Electronics,33,2025-12-31,191,0,NaT,5,2500,4.801437
3,P004,2025-12-31,Travel Backpack,Accessories,23,2025-12-31,202,0,NaT,3,1200,2.772442
4,P005,2025-12-31,Yoga Mat,Fitness,33,2025-12-31,251,0,NaT,6,700,6.277869


In [22]:
SERVICE_LEVEL = 0.95
Z_VALUE = 1.645

inventory_state["safety_stock"] = (
    Z_VALUE
    * inventory_state["error_std"]
    * np.sqrt(inventory_state["lead_time_days"])
)

inventory_state["safety_stock"] = (
    inventory_state["safety_stock"].round(0)
)

inventory_state

,product_id,last_actual_date,product_name,category,last_actual_demand,snapshot_date,current_stock,open_order_qty,expected_arrival_date,lead_time_days,unit_cost,error_std,safety_stock
0,P001,2025-12-31,Wireless Headphones,Electronics,24,2025-12-31,225,0,NaT,4,1000,5.594148,18.0
1,P002,2025-12-31,Running Shoes,Footwear,22,2025-12-31,190,0,NaT,7,1800,3.845135,17.0
2,P003,2025-12-31,Smart Watch,Electronics,33,2025-12-31,191,0,NaT,5,2500,4.801437,18.0
3,P004,2025-12-31,Travel Backpack,Accessories,23,2025-12-31,202,0,NaT,3,1200,2.772442,8.0
4,P005,2025-12-31,Yoga Mat,Fitness,33,2025-12-31,251,0,NaT,6,700,6.277869,25.0


In [24]:
inventory_state = inventory_state.merge(
    lead_time_demand,
    on="product_id",
    how="left"
)

inventory_state

,product_id,last_actual_date,product_name,category,last_actual_demand,snapshot_date,current_stock,open_order_qty,expected_arrival_date,lead_time_days,unit_cost,error_std,safety_stock,lead_time_demand
0,P001,2025-12-31,Wireless Headphones,Electronics,24,2025-12-31,225,0,NaT,4,1000,5.594148,18.0,158.253006
1,P002,2025-12-31,Running Shoes,Footwear,22,2025-12-31,190,0,NaT,7,1800,3.845135,17.0,199.272079
2,P003,2025-12-31,Smart Watch,Electronics,33,2025-12-31,191,0,NaT,5,2500,4.801437,18.0,168.577713
3,P004,2025-12-31,Travel Backpack,Accessories,23,2025-12-31,202,0,NaT,3,1200,2.772442,8.0,64.259758
4,P005,2025-12-31,Yoga Mat,Fitness,33,2025-12-31,251,0,NaT,6,700,6.277869,25.0,249.945099


In [25]:
inventory_state["reorder_point"] = (
    inventory_state["lead_time_demand"]
    + inventory_state["safety_stock"]
)

inventory_state

,product_id,last_actual_date,product_name,category,last_actual_demand,snapshot_date,current_stock,open_order_qty,expected_arrival_date,lead_time_days,unit_cost,error_std,safety_stock,lead_time_demand,reorder_point
0,P001,2025-12-31,Wireless Headphones,Electronics,24,2025-12-31,225,0,NaT,4,1000,5.594148,18.0,158.253006,176.253006
1,P002,2025-12-31,Running Shoes,Footwear,22,2025-12-31,190,0,NaT,7,1800,3.845135,17.0,199.272079,216.272079
2,P003,2025-12-31,Smart Watch,Electronics,33,2025-12-31,191,0,NaT,5,2500,4.801437,18.0,168.577713,186.577713
3,P004,2025-12-31,Travel Backpack,Accessories,23,2025-12-31,202,0,NaT,3,1200,2.772442,8.0,64.259758,72.259758
4,P005,2025-12-31,Yoga Mat,Fitness,33,2025-12-31,251,0,NaT,6,700,6.277869,25.0,249.945099,274.945099


In [26]:
inventory_state["inventory_position"] = (
    inventory_state["current_stock"]
    + inventory_state["open_order_qty"]
)

inventory_state

,product_id,last_actual_date,product_name,category,last_actual_demand,snapshot_date,current_stock,open_order_qty,expected_arrival_date,lead_time_days,unit_cost,error_std,safety_stock,lead_time_demand,reorder_point,inventory_position
0,P001,2025-12-31,Wireless Headphones,Electronics,24,2025-12-31,225,0,NaT,4,1000,5.594148,18.0,158.253006,176.253006,225
1,P002,2025-12-31,Running Shoes,Footwear,22,2025-12-31,190,0,NaT,7,1800,3.845135,17.0,199.272079,216.272079,190
2,P003,2025-12-31,Smart Watch,Electronics,33,2025-12-31,191,0,NaT,5,2500,4.801437,18.0,168.577713,186.577713,191
3,P004,2025-12-31,Travel Backpack,Accessories,23,2025-12-31,202,0,NaT,3,1200,2.772442,8.0,64.259758,72.259758,202
4,P005,2025-12-31,Yoga Mat,Fitness,33,2025-12-31,251,0,NaT,6,700,6.277869,25.0,249.945099,274.945099,251
